In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/df_final.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Cyclic transformation
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Check
print("New columns:")
new_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
print(df[new_cols].describe().round(3))

In [ ]:
# Lag features for fuel prices
df['gas_price_lag_24h'] = df['gas_price'].shift(24)
df['gas_price_lag_168h'] = df['gas_price'].shift(168)

df['coal_price_lag_24h'] = df['coal_price'].shift(24)
df['coal_price_lag_168h'] = df['coal_price'].shift(168)

df['co2_price_lag_24h'] = df['co2_price'].shift(24)

# Sanity check — first 170 rows should have NaN for 168h lags
print("NaNs from fuel lags:")
lag_cols = ['gas_price_lag_24h', 'gas_price_lag_168h',
            'coal_price_lag_24h', 'coal_price_lag_168h',
            'co2_price_lag_24h']
print(df[lag_cols].isnull().sum())

In [ ]:
# Renewable share — ratio of renewables to total generation
# More informative than absolute MW values
df['renewable_share'] = (
    df['wind_onshore'] + df['wind_offshore'] + df['solar']
) / df['total_generation'].replace(0, np.nan)

# Fuel cost index — weighted combo of gas and coal
# Gas is the strongest single predictor from EDA
df['fuel_cost_index'] = df['gas_price'] * 0.6 + df['coal_price'] * 0.4

# Dispatchable generation — thermal + nuclear (non-renewable, price-setting)
df['dispatchable_gen'] = (
    df['coal_generation'] + df['gas_generation'] + df['nuclear_generation']
)

# Demand-supply balance — load minus total generation
# Negative = oversupply (pushes price down), positive = undersupply
df['demand_supply_gap'] = df['load'] - df['total_generation']

# Sanity check
check_cols = ['renewable_share', 'fuel_cost_index',
              'dispatchable_gen', 'demand_supply_gap']
print(df[check_cols].describe().round(2))

In [ ]:
# Peak hour flag (morning 7-10, evening 17-21)
df['is_peak_hour'] = df['hour'].isin(range(7, 11)) | df['hour'].isin(range(17, 22))
df['is_peak_hour'] = df['is_peak_hour'].astype(int)

# Wind suppression during peak — high wind during peak hours
# This is when wind has biggest price impact (merit order effect)
df['wind_x_peak'] = (
    (df['wind_onshore'] + df['wind_offshore']) * df['is_peak_hour']
)

# Gas price pressure during peak demand
df['gas_x_peak'] = df['gas_price'] * df['is_peak_hour']

# Solar suppression — solar during high demand hours
df['solar_x_demand'] = df['solar'] * df['load']

# Renewable share during peak — captures merit order effect
df['renewable_share_x_peak'] = df['renewable_share'] * df['is_peak_hour']

# Sanity check
interact_cols = ['is_peak_hour', 'wind_x_peak', 'gas_x_peak',
                 'solar_x_demand', 'renewable_share_x_peak']
print(df[interact_cols].describe().round(2))
print(f"\nPeak hours: {df['is_peak_hour'].sum():,} rows ({df['is_peak_hour'].mean()*100:.1f}%)")

In [ ]:
# Energy crisis period flag (2021-2023 gas/energy crisis)
df['is_crisis_period'] = (
    (df['timestamp'] >= '2021-01-01') &
    (df['timestamp'] <= '2023-12-31')
).astype(int)

# High price regime — above 75th percentile historically
price_75 = df['price'].quantile(0.75)
df['is_high_price_regime'] = (df['price'] > price_75).astype(int)

# Negative price flag
df['is_negative_price'] = (df['price'] < 0).astype(int)

# Year feature — captures long-term trend (energy transition)
df['year'] = df['timestamp'].dt.year

# Sanity check
print(f"Crisis period rows:    {df['is_crisis_period'].sum():,} ({df['is_crisis_period'].mean()*100:.1f}%)")
print(f"High price regime:     {df['is_high_price_regime'].sum():,} ({df['is_high_price_regime'].mean()*100:.1f}%)")
print(f"Negative price rows:   {df['is_negative_price'].sum():,} ({df['is_negative_price'].mean()*100:.1f}%)")
print(f"Years in dataset:      {sorted(df['year'].unique())}")

In [ ]:
# Check all NaNs before dropping
print("NaNs per column before drop:")
nan_cols = df.columns[df.isnull().any()].tolist()
print(df[nan_cols].isnull().sum().sort_values(ascending=False))

# Drop first 168 rows — covers all lag-induced NaNs
df = df.dropna(subset=nan_cols).reset_index(drop=True)

# Verify
print(f"\nShape after drop: {df.shape}")
print(f"Remaining NaNs:   {df.isnull().sum().sum()}")
print(f"New date range:   {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

In [ ]:
EXCLUDE_COLS = [
    'timestamp',
    'price',
    'is_high_price_regime',
    'is_negative_price',
    'actual_wind_offshore',
    'actual_wind_onshore',
    'actual_solar',
    'actual_load',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Total columns:   {df.shape[1]}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded:        {len(EXCLUDE_COLS)}")
print(f"\nFeatures going into model:")
for col in feature_cols:
    print(f"  {col}")

""" 
    df.to_csv('../data/df_features.csv', index=False)
    print(f"Saved: df_features.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)")
 """

In [ ]:
# Residual Load
df['residual_load'] = df['load'] - (df['wind_onshore'] + df['wind_offshore'] + df['solar'])

# Load Ramp
df['load_ramp'] = df['load'].diff(1)

# Renewable Ramp
res = df['wind_onshore'] + df['wind_offshore'] + df['solar']
df['renewable_ramp'] = res.diff(1)

# Price Volatility 
df['price_volatility_24h'] = df['price'].rolling(24).std()

# Δ Wind Forecast
# wind_onshore i wind_offshore su već forecasted vrijednosti
df['total_wind_forecast'] = df['wind_onshore'] + df['wind_offshore']
df['delta_wind_forecast'] = df['total_wind_forecast'] - df['total_wind_forecast'].shift(24)

In [ ]:
df.head()

In [ ]:
#checking the data

In [ ]:
print(df[['residual_load', 'load_ramp', 'renewable_ramp', 
          'price_volatility_24h', 'delta_wind_forecast']].isnull().sum())

print(f"\nUkupno NaN-ova: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

In [ ]:
# losing first 24 rows

df = df.dropna(subset=['load_ramp', 'renewable_ramp', 
                        'price_volatility_24h', 'delta_wind_forecast']).reset_index(drop=True)

print(f"Shape after drop: {df.shape}")
print(f"Nan's: {df.isnull().sum().sum()}")
print(f"New date range: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

In [ ]:
df.to_csv('../data/df_features.csv', index=False)
print(f"Saved: df_features.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)")

In [ ]:
# MODELING